# 04 · Modelado — Regresión Lineal (baseline) + XGBoost
**Diplomado en Ciencia de Datos · ENES UNAM León**

### Estrategia de evaluación para series temporales

> ⚠️ **Importante:** En series temporales **NO se usa `train_test_split` aleatorio**. Si mezclas fechas futuras en el entrenamiento, el modelo 'hace trampa' (data leakage). Siempre dividimos cronológicamente:

```
|------- Train (70%) -------|-- Val (15%) --|-- Test (15%) --|
2024-01                  ~Oct 2024       ~Dec 2024       2025-03
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Cargar dataset con features
df = pd.read_csv('../data/processed/sol_usd_features.csv', index_col=0, parse_dates=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Dataset cargado: {df.shape}')
df.head(3)

## 4.1 Definir features y target

In [ ]:
FEATURE_COLS = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_7', 'Close_lag_14',
    'SMA_7', 'SMA_21', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width',
    'volatility_21d', 'Volume_change', 'Volume_SMA7',
    'day_of_week', 'month'
]
TARGET = 'Target'

X = df[FEATURE_COLS]
y = df[TARGET]

print(f'Features : {X.shape[1]}')
print(f'Muestras : {X.shape[0]}')
print(f'Target   : precio de cierre del día siguiente (USD)')

## 4.2 Split cronológico (sin data leakage)

In [ ]:
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X.iloc[:train_end],  y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],    y.iloc[val_end:]

dates_train = df.index[:train_end]
dates_val   = df.index[train_end:val_end]
dates_test  = df.index[val_end:]

print(f'Train : {len(X_train)} muestras | {dates_train[0].date()} → {dates_train[-1].date()}')
print(f'Val   : {len(X_val)}   muestras | {dates_val[0].date()}   → {dates_val[-1].date()}')
print(f'Test  : {len(X_test)}  muestras | {dates_test[0].date()}  → {dates_test[-1].date()}')

# Visualizar el split
plt.figure(figsize=(14, 3))
plt.plot(dates_train, y_train, label='Train', color='#9945FF', linewidth=1.5)
plt.plot(dates_val,   y_val,   label='Validación', color='#FFB800', linewidth=1.5)
plt.plot(dates_test,  y_test,  label='Test', color='#14F195', linewidth=1.5)
plt.axvline(dates_val[0],  color='white', linestyle='--', alpha=0.5)
plt.axvline(dates_test[0], color='white', linestyle='--', alpha=0.5)
plt.title('División temporal del dataset')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_split.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.3 Normalización

La regresión lineal requiere que las features estén en la misma escala. XGBoost no lo requiere, pero no daña aplicarlo.

In [ ]:
scaler = StandardScaler()

# IMPORTANTE: fit solo en train, transform en val y test
# Esto evita data leakage al normalizar
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Normalización aplicada ✓')
print(f'Media de features (train): {X_train_sc.mean(axis=0).mean():.6f} (≈0 esperado)')
print(f'Desv. std (train)         : {X_train_sc.std(axis=0).mean():.4f} (≈1 esperado)')

## 4.4 Función de métricas

In [ ]:
def evaluar_modelo(nombre, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f'\n📊 {nombre}')
    print(f'  MAE  = ${mae:.2f}   (error absoluto promedio)')
    print(f'  RMSE = ${rmse:.2f}  (penaliza errores grandes)')
    print(f'  MAPE = {mape:.2f}%  (error porcentual promedio)')
    print(f'  R²   = {r2:.4f}    (1.0 = perfecto)')
    return {'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

resultados = []

## 4.5 Modelo 1: Regresión Lineal (Baseline)

El baseline es el punto de referencia mínimo. Si XGBoost no supera este modelo simple, algo está mal.

In [ ]:
lr = Ridge(alpha=1.0)
lr.fit(X_train_sc, y_train)

pred_lr_val  = lr.predict(X_val_sc)
pred_lr_test = lr.predict(X_test_sc)

res_lr_val  = evaluar_modelo('Ridge Regression (Validación)', y_val, pred_lr_val)
res_lr_test = evaluar_modelo('Ridge Regression (Test)', y_test, pred_lr_test)
resultados.append(res_lr_test)

In [ ]:
# Visualizar predicciones en test
plt.figure(figsize=(14, 5))
plt.plot(dates_test, y_test.values, label='Real', color='#9945FF', linewidth=1.5)
plt.plot(dates_test, pred_lr_test, label='Predicción Ridge', color='#FFB800', linewidth=1.5, linestyle='--')
plt.title('Ridge Regression — Predicción vs Real (Test set)')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_ridge_prediccion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Modelo 2: XGBoost

Gradient Boosting basado en árboles. Muy usado en competencias de datos y en finanzas por su capacidad de capturar relaciones no lineales.

In [ ]:
# Entrenamiento con early stopping usando el set de validación
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric='rmse',
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f'Mejor iteración (early stopping): {xgb_model.best_iteration}')

In [ ]:
pred_xgb_val  = xgb_model.predict(X_val)
pred_xgb_test = xgb_model.predict(X_test)

res_xgb_val  = evaluar_modelo('XGBoost (Validación)', y_val, pred_xgb_val)
res_xgb_test = evaluar_modelo('XGBoost (Test)', y_test, pred_xgb_test)
resultados.append(res_xgb_test)

In [ ]:
# Visualizar predicciones en test
plt.figure(figsize=(14, 5))
plt.plot(dates_test, y_test.values, label='Real', color='#9945FF', linewidth=1.5)
plt.plot(dates_test, pred_xgb_test, label='Predicción XGBoost', color='#14F195', linewidth=1.5, linestyle='--')
plt.title('XGBoost — Predicción vs Real (Test set)')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_xgboost_prediccion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Feature Importance (XGBoost)

In [ ]:
importance = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#9945FF' if i >= len(importance) - 5 else '#555' for i in range(len(importance))]
importance.plot(kind='barh', color=colors)
plt.title('XGBoost — Importancia de Features')
plt.xlabel('Importancia')
plt.tight_layout()
plt.savefig('../data/raw/04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 features más importantes:')
print(importance.tail(5).sort_values(ascending=False))

## 4.8 Comparación final de modelos

In [ ]:
df_resultados = pd.DataFrame(resultados)
df_resultados.set_index('modelo', inplace=True)
print('\n===== COMPARACIÓN DE MODELOS (Test Set) =====')
print(df_resultados.round(4).to_string())

# Gráfica comparativa
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    df_resultados[metric].plot(kind='bar', ax=ax, color=['#FFB800', '#14F195'], edgecolor='none')
    ax.set_title(metric)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    ax.set_ylabel('USD' if metric != 'MAPE' else '%')

plt.suptitle('Comparación de métricas — Test Set', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/raw/04_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Predicciones lado a lado en test
plt.figure(figsize=(14, 5))
plt.plot(dates_test, y_test.values,    label='Real',    color='#9945FF', linewidth=2)
plt.plot(dates_test, pred_lr_test,     label='Ridge',   color='#FFB800', linewidth=1.5, linestyle='--', alpha=0.8)
plt.plot(dates_test, pred_xgb_test,    label='XGBoost', color='#14F195', linewidth=1.5, linestyle='--', alpha=0.8)
plt.title('Comparación de modelos — Test Set')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/04_comparacion_predicciones.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.9 Guardar modelos y predicciones

In [ ]:
import pickle, os
os.makedirs('../data/processed', exist_ok=True)

# Guardar modelos
with open('../data/processed/ridge_model.pkl', 'wb') as f:
    pickle.dump(lr, f)
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
xgb_model.save_model('../data/processed/xgboost_model.json')

# Guardar predicciones para el notebook de evaluación
df_preds = pd.DataFrame({
    'date': dates_test,
    'real': y_test.values,
    'pred_ridge': pred_lr_test,
    'pred_xgb': pred_xgb_test
}).set_index('date')
df_preds.to_csv('../data/processed/predicciones_test.csv')

print('Modelos y predicciones guardados ✓')

---
## ✅ Resumen del modelado

| Modelo | MAE | RMSE | MAPE | R² |
|---|---|---|---|---|
| Ridge Regression | *ver output* | *ver output* | *ver output* | *ver output* |
| XGBoost | *ver output* | *ver output* | *ver output* | *ver output* |

**Siguiente paso →** `05_evaluation.ipynb`